# Import Module

In [ ]:
# Import module
import json
import spacy
from transformers import AutoModelForQuestionAnswering, AutoTokenizer, pipeline
import re

# Load data from shortcut database

In [ ]:
# Load data
with open("shortcut_database.json", "r") as f:
    data = json.load(f)

# Extract Definition

In [2]:
# Load NLP model (scispaCy for medical texts)
nlp = spacy.load("en_core_sci_lg") 

# Load QA model & tokenizer
model_name = "deepset/roberta-base-squad2"
model = AutoModelForQuestionAnswering.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Create a question-answering pipeline
qa_pipeline = pipeline('question-answering', model=model, tokenizer=tokenizer)

c:\Users\Thong Vo\anaconda3\envs\TfGPUPy310\lib\site-packages\spacy\language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


In [ ]:
# Function to extract definitions
def extract_definition(term, content):
    # Use SpaCy to process text
    doc = nlp(content)
    sentences = [sent.text for sent in doc.sents]
    
    # Use transformer QA model for definition extraction
    query = f"What is {term}?"
    definition = ""
    for sentence in sentences:
        result = qa_pipeline(question=query, context=sentence)
        if result['score'] > 0.3:  # Filter by confidence score
            definition = definition + " " + sentence
    if (len(definition) == 0):
        definition = "Definition not found"
    return definition

In [ ]:
# Extract definition feature from data
for item in data:
    term = item["term"]
    content = item['content']
    content = re.sub(r'\n', ' ', content)
    item["definition"] = extract_definition(term, content)

# Extract other features (Monir's Code)

In [12]:
# Load SciSpaCy for medical term extraction
try:
    nlp = spacy.load("en_core_sci_lg")
except OSError as e:
    print(f"Error loading SciSpaCy model: {e}")
    exit()

# Load the summarization pipeline and tokenizer
try:
    summarizer = pipeline("summarization", model="facebook/bart-large-cnn", tokenizer="facebook/bart-large-cnn")
    tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
except Exception as e:
    print(f"Error loading summarization pipeline: {e}")
    exit()

c:\Users\Thong Vo\anaconda3\envs\TfGPUPy310\lib\site-packages\spacy\language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


In [13]:
# Function to tokenize and chunk text
def tokenize_and_chunk(text, max_tokens=1024):
    """Tokenize text and split into chunks within token limit."""
    tokens = tokenizer(text, return_tensors="pt", truncation=False, add_special_tokens=False)["input_ids"][0]
    chunks = []
    for i in range(0, len(tokens), max_tokens - 50):  # Leave buffer for summarization tokens
        chunk = tokens[i:i + max_tokens - 50]
        chunks.append(tokenizer.decode(chunk, skip_special_tokens=True))
    return chunks

In [14]:
# Function to summarize long text
def summarize_text(text):
    """Summarize text by splitting it into manageable chunks."""
    chunks = tokenize_and_chunk(text, max_tokens=1024)
    summaries = []
    for chunk in chunks:
        try:
            summary = summarizer(chunk, max_length=60, min_length=20, do_sample=False)[0]["summary_text"]
            summaries.append(summary)
        except Exception as e:
            print(f"Error summarizing chunk: {e}")
            summaries.append("Error summarizing this chunk.")
    return " ".join(summaries)

In [15]:
# Function to extract relevant sections
def extract_relevant_sections(text, term):
    """Extract sections related to symptoms, diagnosis, treatment, and risk factors."""
    sections = {"symptoms": "", "diagnosis": "", "treatment": "", "risk_factors": ""}
    sentences = [sent.text for sent in nlp(text).sents if term.lower() in sent.text.lower()]

    # Keyword-based extraction
    for sent in sentences:
        if any(keyword in sent.lower() for keyword in ["symptom", "sign"]):
            sections["symptoms"] += sent + " "
        elif any(keyword in sent.lower() for keyword in ["diagnosis", "diagnose", "diagnostic"]):
            sections["diagnosis"] += sent + " "
        elif any(keyword in sent.lower() for keyword in ["treatment", "therapy", "medication", "care"]):
            sections["treatment"] += sent + " "
        elif any(keyword in sent.lower() for keyword in ["risk", "factor", "complication"]):
            sections["risk_factors"] += sent + " "

    # Summarize each section
    for key, value in sections.items():
        if value.strip():
            sections[key] = summarize_text(value)

    return sections

In [ ]:
# Extract other features
for item in data:
    term = item["term"]
    content = item["content"]
    # Ensure content is a string
    if isinstance(content, list):
        content = " ".join(content)
    elif not isinstance(content, str):
        content = str(content)

    # Extract relevant sections
    relevant_sections = extract_relevant_sections(content, term)
    
    for key, value in relevant_sections.items():
        item[key] = value

Your max_length is set to 60, but your input_length is only 19. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=9)
Your max_length is set to 60, but your input_length is only 19. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=9)
Your max_length is set to 60, but your input_length is only 14. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=7)
Your max_length is set to 60, but your input_length is only 20. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=10)
Your max_le

# Save data to extracted database

In [ ]:
# Save extracted data to json file
extracted_data = []
for item in data:
    new_dict = {}
    for key, value in item.items():
        if key in ["term", "definition", "symptoms", "diagnosis", "treatment", "risk_factors"]:
            new_dict[key] = value
    extracted_data.append(new_dict)

with open('extracted_database.json', 'w') as fo:
    json.dump(extracted_data, fo)